In [ ]:
# =============================================
# Ecommerce Analytics Project – Python Analysis
# =============================================

# Step 1: Import Libraries
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


In [11]:
# ------------------------------
# Step 2: Load Data
# ------------------------------
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
products = pd.read_csv('products.csv')
marketing = pd.read_csv('marketing_campaigns.csv')
feedback = pd.read_csv('customer_feedback.csv')

In [12]:
# ------------------------------
# Step 3: Data Cleaning / Preprocessing
# ------------------------------
# Convert date columns
orders['order_date'] = pd.to_datetime(orders['order_date'])
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
marketing['start_date'] = pd.to_datetime(marketing['start_date'])
marketing['end_date'] = pd.to_datetime(marketing['end_date'])
feedback['feedback_date'] = pd.to_datetime(feedback['feedback_date'])

In [13]:
# Filter only delivered orders
orders = orders[orders['order_status'] == 'Delivered']

In [14]:
# ------------------------------
# Step 4: RFM Analysis (Customer Segmentation)
# ------------------------------
# Recency = days since last purchase
snapshot_date = orders['order_date'].max() + pd.Timedelta(days=1)
rfm = orders.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'order_amount': 'sum'
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']

# Segment customers into quartiles
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

# Combine scores
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Map to segments
def rfm_segment(score):
    if score.startswith('4'):
        return 'Champions'
    elif score.startswith('3'):
        return 'Loyal'
    elif score.startswith('2'):
        return 'Potential'
    else:
        return 'At Risk'

rfm['Segment'] = rfm['RFM_Score'].apply(rfm_segment)

print("Top 5 RFM customers:")
print(rfm.head())



Top 5 RFM customers:
   customer_id  Recency  Frequency  Monetary R_Score F_Score M_Score  \
0         1001       10          6   2987.55       4       4       4   
1         1002      145          1    214.88       2       1       1   
2         1003       35          5   1928.05       4       3       2   
3         1004      254          3   1824.58       1       1       2   
4         1005       29          9   4336.25       4       4       4   

  RFM_Score    Segment  
0       444  Champions  
1       211  Potential  
2       432  Champions  
3       112    At Risk  
4       444  Champions  


In [15]:
# ------------------------------
# Step 5: Sales Aggregations
# ------------------------------
# Revenue by Month
orders['month'] = orders['order_date'].dt.to_period('M')
monthly_sales = orders.groupby('month').agg(
    total_revenue=('order_amount','sum'),
    total_orders=('order_id','count'),
    avg_order_value=('order_amount','mean')
).reset_index()


In [16]:
# --- ANALYSIS 1: Sales Trends (Monthly/Quarterly) ---
orders['month'] = orders['order_date'].dt.to_period('M')
orders['quarter'] = orders['order_date'].dt.to_period('Q')

In [17]:
# Aggregate Delivered Revenue by Month
monthly_sales = orders[orders['order_status'] == 'Delivered'].groupby('month')['order_amount'].agg(['sum', 'count'])
monthly_sales.rename(columns={'sum': 'Total Revenue', 'count': 'Total Orders'}, inplace=True)

In [18]:
# --- ANALYSIS 2: Customer Segmentation (Top 20%) ---
# Calculate Revenue per Customer
cust_rev = orders[orders['order_status'] == 'Delivered'].groupby('customer_id')['order_amount'].sum().reset_index()

In [19]:
# Use qcut to divide into 5 groups (Quintiles). Group '1' is the top 20%.
cust_rev['revenue_quintile'] = pd.qcut(cust_rev['order_amount'], 5, labels=[5, 4, 3, 2, 1])
top_20_pct_customers = cust_rev[cust_rev['revenue_quintile'] == 1].merge(customers, on='customer_id')

In [20]:
# --- ANALYSIS 3: Repeat Purchase Rate by Segment ---
order_counts = orders.groupby('customer_id')['order_id'].count().reset_index()
order_counts['is_repeat'] = order_counts['order_id'] > 1

In [21]:
# Merge with Customer table to get segments
segment_data = pd.merge(customers, order_counts, on='customer_id')
repeat_rate_by_segment = segment_data.groupby('customer_segment')['is_repeat'].mean() * 100

In [22]:
# --- ANALYSIS 4: Marketing Campaign ROI & Channel Impact ---
marketing['conversion_rate'] = (marketing['conversions'] / marketing['clicks']) * 100
marketing['cost_per_conversion'] = marketing['budget'] / marketing['conversions']

In [23]:
# Group by Channel to see performance
channel_performance = marketing.groupby('channel')[['conversion_rate', 'cost_per_conversion']].mean()

In [24]:
# --- ANALYSIS 5: Product Performance & Sentiment ---
# Link products to revenue via the feedback bridge (as done in SQL)
product_bridge = pd.merge(feedback, products, on='product_id')
revenue_bridge = pd.merge(product_bridge, orders[orders['order_status'] == 'Delivered'], on='customer_id')

In [25]:
# Top 10 Products by Revenue
top_10_products = revenue_bridge.groupby('product_name')['order_amount'].sum().sort_values(ascending=False).head(10)

In [ ]:
# Quick Sentiment Classification
def simple_sentiment(text):
    text = str(text).lower()
    if any(w in text for w in ['great', 'excellent', 'love']): return 'Positive'
    if any(w in text for w in ['bad', 'poor', 'disappointed']): return 'Negative'
    if any(w in text for w in ['slow', 'delay']): return 'Shipping Issue'
    return 'Neutral'

feedback['sentiment'] = feedback['review'].apply(simple_sentiment)
sentiment_summary = feedback.groupby('sentiment')['rating'].agg(['count', 'mean'])

In [26]:
# --- OUTPUT RESULTS ---
print("--- Sales Trends (First 5 Months) ---")
print(monthly_sales.head())
print("\n--- Channel Performance (Efficiency) ---")
print(channel_performance.sort_values(by='conversion_rate', ascending=False))
print("\n--- Repeat Purchase Rate (%) by Segment ---")
print(repeat_rate_by_segment)

--- Sales Trends (First 5 Months) ---
         Total Revenue  Total Orders
month                               
2024-03        3231.59             8
2024-04       13928.20            27
2024-05       21101.98            38
2024-06       19996.58            41
2024-07       20064.35            36

--- Channel Performance (Efficiency) ---
           conversion_rate  cost_per_conversion
channel                                        
Social           71.750460             6.615480
Email            28.010428            17.005365
Paid Ads         20.066888            12.366729
Affiliate        13.763331            34.753407

--- Repeat Purchase Rate (%) by Segment ---
customer_segment
Bronze    90.000000
Gold      89.473684
Silver    98.387097
Name: is_repeat, dtype: float64


In [27]:
 #---Churn Prediction (Basic Model)---
# ------------------------------
# Simple churn = no orders in last 90 days
orders_last = orders.groupby('customer_id')['order_date'].max().reset_index()
orders_last['churn'] = (snapshot_date - orders_last['order_date']).dt.days > 90

# Merge features
churn_data = orders_last.merge(rfm[['customer_id','Recency','Frequency','Monetary']], on='customer_id')

X = churn_data[['Recency','Frequency','Monetary']]
y = churn_data['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\nChurn Prediction Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



Churn Prediction Accuracy: 1.0
              precision    recall  f1-score   support

       False       1.00      1.00      1.00        13
        True       1.00      1.00      1.00        25

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38

